<table style="width:100%">
  <tr>
    <td valign="top"><img src="../data/img/FER_logo_2.png" width=300 height=80 align="left"></td>
    <td valign="top"><img src="../data/img/LARES_2_transparent.png" width=250 height=80 align="right"></td>
  </tr>
 </table>

# Exploratory data analysis (EDA) and data processing
In today's notebook we will go through the EDA and data processing process of two separate datasets:
- the Ames Housing dataset, and
- the Iris dataset

For each of the datasets will go through the same structure:
- load the data,
- check basic properties,
- EDA,
- process the dataset.

Throughout all the steps we will keep in mind that we are analyzing this data towards building regression/classification machine learning models.
Additionally, for the Housing dataset, we will explain some basic feature selection techniques.

## Import libraries
Import all the necessary Python libraries.
Set some general options for plotting, suppress warnings etc.

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import norm

In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")
import matplotlib.pyplot as plt
tex_fonts = {
    "font.family": "serif",
    # Use 26pt font in plots
    "axes.labelsize": 20,
    "font.size": 20,
    "figure.titlesize": 20,
    # Make the legend/label fonts a little smaller
    "legend.title_fontsize": 18,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18
}

plt.rcParams.update(tex_fonts)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Ames Housing dataset

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this playground dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence.

With 79 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, our **goal is to predict the final price of each home**.

The Ames Housing dataset was compiled by Dean De Cock for use in data science education. It's an incredible alternative for data scientists looking for a modernized and expanded version of the often cited Boston Housing dataset. The dataset, along many exemplary notebooks can be found on [Kaggle](https://www.kaggle.com/c/house-prices-advanced-regression-techniques). Keep in mind, the Kaggle version of the dataset keeps 50% of house prices hidden for scoring purposes - full version of the dataset can be found [here](https://www.kaggle.com/datasets/prevek18/ames-housing-dataset).

![title](../data/img/house_prices.jpg)

## Load dataset

In [ ]:
# load the stored dataset from the data/ folder and print out few of the first rows for easier visual checking.
df = pd.read_csv('../data/housing_prices/housing.csv',index_col='Id')
df.head()

In [ ]:
# dataframe properties
df.info()

## Part 1 - Check basic dataset properties
Check some basic properties of the dataset to get a better sense of the data we are dealing with such as:
- check the size of the dataset (rows - data recordings, columns - inputs/features)
- check what are the available inputs/features (column names)
- check the types of available inputs/features
- check some basic statistical properties of the data
- missing values

In [ ]:
# dataset size (rows, columns)
print('Size of dataset', df.shape)

In [ ]:
# dataset columns (possible features)
print(len(df.columns))
print(df.columns.values)

In [ ]:
# dataset column types
df.dtypes.value_counts()

In [ ]:
# basic dataset statistics
df.describe()

In [ ]:
# make it a little bit easier on the eyes :)
df.describe().transpose()

**Conclusions**:
- *lot of possible features (80), 37 numeric, 43 categorical: doesn't seem likely that we will use them all,*
- *different ranges within the feature set (e.g. LotArea [1300, 215245], OverallQual [1, 10]),*
- *missing values for some features.*

### Missing values
Check and handle (remove or fill - if possible) missing values of data.

In [ ]:
# find the columns with the largest amounts of missing data
total = df.isnull().sum().sort_values(ascending = False)
percent = (df.isnull().sum()/df.isnull().count()*100).sort_values(ascending = False)
missing_data  = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data.head(20)

In [ ]:
# remove all features with over 0.5% of missing data
df = df.drop((missing_data[missing_data['Percent'] > 0.5]).index, axis=1)
# remove the single data with missing 'Electrical' data
df = df.drop(df.loc[df['Electrical'].isnull()].index)
# check the new dataset size
print('Size of dataset', df.shape)

**Conclusions**:
- *large number of sparse (unusablle) features,*
- *removed 18 features and one row with missing Electrical.*

## Part 2 - Exploratory data analysis (EDA)
Take a closer look into the available dataset:
- analyze the variable we are trying to predict in more detail (basic statistics, distribution)
- correlations between variables
- outliers?
- closer look into the most likely features

### Sale Price variable

In [ ]:
# basic statistics (again)
df['SalePrice'].describe()

In [ ]:
# scatter plot for all sales price data
plt.figure(figsize=(15,6))
plt.scatter(df.index,df['SalePrice'], s=50)
plt.xlim(0,1500)
plt.xlabel('Id')
plt.ylabel('SalePrice');

In [ ]:
#histogram and normal probability plot
plt.figure(figsize=(15,6))
sns.distplot(df['SalePrice'], fit=norm);

In [ ]:
#skewness and kurtosis
print("Skewness: %f" % df['SalePrice'].skew()) 
print("Kurtosis: %f" % df['SalePrice'].kurt())

The histogram is an effective graphical technique for showing both the skewness and kurtosis of data set.

Skewness is a measure of symmetry, or more precisely, the lack of symmetry. The skewness for a normal distribution is zero, and any symmetric data should have a skewness near zero (negative - skewed left, positive - skewed right).

Kurtosis is a measure of whether the data are heavy-tailed or light-tailed relative to a normal distribution. The kurtosis for a standard normal distribution is three. Pandas uses Fisher’s definition of kurtosis (kurt-3).

**Conclusions**:
- *skewed distribution of the target variable*,
- *outliers present*,
- *some houses with very high prices.*

### Correlations
Correlation coefficients are used to measure how strong a relationship is between two variables. There are several types of correlation coefficient, but the most popular is Pearson’s.

In [ ]:
# calculate and plot all correlations...
corr = df.corr(numeric_only=True)
plt.figure(figsize=(12,10))
sns.heatmap(corr, xticklabels=corr.columns, yticklabels=corr.columns);

In [ ]:
# sort the correlations between features and Sale price
corr['SalePrice'].sort_values(ascending=False)

In [ ]:
# plot correlations of the 10 most correlated features
relevant_inp = np.abs(corr['SalePrice']).sort_values(ascending=False).index[0:11]
relevant_corr = df[relevant_inp].corr(numeric_only=True)

# create a mask for the lower corr triangle only
mask = np.zeros_like(relevant_corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

# want diagonal elements as well
mask[np.diag_indices_from(mask)] = False

# plot the corr heatmap
plt.figure(figsize=(9,7))
sns.heatmap(relevant_corr, xticklabels=relevant_corr.columns, yticklabels=relevant_corr.columns,
        mask=mask, square=True, annot=True, linewidths=.5, cbar_kws={"shrink": .8}, annot_kws={"fontsize":14});

**Conclusions**:
- *a lot of uncorrelated variables that can be omitted,*
- *high correlation with the expected variables (common knowledge): overall house quality, living area, garage, bathroom, year built and remodelled...*
- *some variable highly correlate among themselves and perhaps can be merged or reduced (garage area and no. of garage cars).*

### Outliers
In statistics, an outlier is a data point that differs significantly from other observations.

In [ ]:
# remember calculated correlations
relevant_inp[1:]

In [ ]:
# plot pairwise relationships in a dataset
g = sns.pairplot(df[['SalePrice','OverallQual','GrLivArea','GarageArea','YearBuilt']])
g.fig.set_size_inches((10,10))

In [ ]:
# plot box plots of sales prices with respect to the house overall quality ratings
plt.figure(figsize=(8,6))
g = sns.boxplot(x=df['OverallQual'],y=df['SalePrice'])

A box plot (or box-and-whisker plot) shows the distribution of quantitative data in a way that facilitates comparisons between variables or across levels of a categorical variable. The box shows the quartiles of the dataset while the whiskers extend to show the rest of the distribution, except for points that are determined to be “outliers” using a method that is a function of the inter-quartile range.

In [ ]:
# box plots of sales prices with respect to house built year
plt.figure(figsize=(25,8))
g = sns.boxplot(x=df['YearBuilt'],y=df['SalePrice'], order=np.sort(df['YearBuilt'].unique()))
plt.xticks(rotation=45, ha='right');

**Conclusions**:
- *outliers present in most of the highly correlated features,*
- *since visual analysis shows good results, remove outliers manualy (e.g., two houses with almost largest living areas and bellow average prices),*
- *further investigate suspicious data and try to find out if there are some other reasons that are not shown in bivariate analysis (e.g., were the oldest houses with high prices been recently remodelled, or have large living areas, etc.).*

### Conclusions - overall:
- *lot of possible features (79), 36 numeric, 43 categorical: doesn't seem likely that will use them all,*
- *different ranges within the feature set (e.g. LotArea [1300, 215245], OverallQual [1, 10]),*
- *missing values for some features,*
- *large number of sparse (unusablle) features,*
- *removed 18 features and one row with missing Electrical,*
- *skewed distribution of the target variable*,
- *outliers present*,
- *some houses with very high prices,*
- *a lot of uncorrelated variables that can be omitted,*
- *high correlation with the expected variables (common knowledge): overall house quality, living area, garage, bathroom, year built and remodelled...*
- *some variable highly correlate among themselves and perhaps can be merged or reduced (garage area and no. of garage cars),*
- *outliers present in most of the highly correlated features,*
- *since visual analysis shows good results, remove outliers manualy (e.g., two houses with almost largest living areas and bellow average prices),*
- *further investigate suspicious data and try to find out if there are some other reasons that are not shown in bivariate analysis (e.g., were the oldest houses with high prices been recently remodelled, or have large living areas, etc.).*

## Part 3 - Process the training dataset
Once the available dataset is analysed with more detail, transform it into the appropriate form suitable for learning various models:
- categorical data encoding
- scaling/transforming
- splitting into train/val/test datasets

### Categorical data

In [ ]:
# scikit-learn categorical variables encoders
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [ ]:
# check if there are any categorical features in the most correlated ones
df[relevant_inp[1:]].head()

In [ ]:
# find other categorical features
categorical_var = df.select_dtypes(include='object')
categorical_var.columns

In [ ]:
# let's see the 'Heating' categories
df['Heating'].unique()

In [ ]:
# let's see the 'MSZoning' categories
df['MSZoning'].unique()

#### Label encoder

In [ ]:
# use scikit-learn label encoder and create an instance for the 'Heating' categories
le = LabelEncoder()
aux = df['Heating']
aux = le.fit_transform(aux)
np.unique(aux)

In [ ]:
# print the value-category map
le.classes_

#### One-hot encoder

In [ ]:
# use scikit-learn one-hot encoder and create an instance for both 'Heating' and 'MSZoning' categories
enc = OneHotEncoder(handle_unknown='ignore')
aux = df[['Heating','MSZoning']]
df_aux = enc.fit_transform(aux)
df_aux

### Dataset splitting
First we split the data into inputs (features) and outputs (targets).

Then we further split into the train-val-test parts of the dataset. The train-val-test split procedure is used to estimate the performance of machine learning algorithms when they are used to make predictions on data not used to train the model.

In [ ]:
# split into intputs and outputs
# use only the 10 most correlated features (simplification for presentation purposes)
X = df[relevant_inp[1:]]
y = df[relevant_inp[0]]

In [ ]:
# use scikit-learn dataset splitter
from sklearn.model_selection import train_test_split

# split the dataset into train - test dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=1)

# separate dataset for validation?
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, shuffle=True, random_state=1) # 0.25 x 0.8 = 0.2

print('Train dataset size:', X_train.shape[0])
print('Validation dataset size:', X_val.shape[0])
print('Test dataset size:', X_test.shape[0])

In [ ]:
# keep only the original train-test split for future use
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=1)

print('Train dataset size:', X_train.shape[0])
print('Test dataset size:', X_test.shape[0])

### Dataset scaling/transforming
Some machine learning algorithms are sensitive to feature scaling while others are virtually invariant to it. 

Machine learning algorithms like linear regression, logistic regression, neural network, etc. that use gradient descent as an optimization technique require data to be scaled. Distance algorithms like KNN, K-means, and SVM are most affected by the range of features. This is because behind the scenes they are using distances between data points to determine their similarity.

Distance algorithms like KNN, K-means, and SVM are most affected by the range of features. This is because behind the scenes they are using distances between data points to determine their similarity.

**NOTE:** we will proceed with the overall dataset (train and test parts together) only for presentation purposes. Additionally, we will not save the scaled versions of the data since the scaling is model dependent as said above.

In [ ]:
# import scikit-learn scalers
from sklearn.preprocessing import MinMaxScaler, StandardScaler, QuantileTransformer, PowerTransformer

In [ ]:
# log transformation
aux = np.log(df['SalePrice'])
plt.figure(figsize=(8,6))
sns.distplot(aux, fit=norm);

In [ ]:
# scikit-learn scalers transformations and plotting
plt.figure(figsize=(8,6))
sns.distplot(MinMaxScaler().fit_transform(df['SalePrice'].values.reshape(-1, 1)),label='MinMax')
sns.distplot(StandardScaler().fit_transform(df['SalePrice'].values.reshape(-1, 1)),label='Standard')
sns.distplot(QuantileTransformer(output_distribution='uniform').fit_transform(df['SalePrice'].values.reshape(-1, 1)),label='Quantile')
sns.distplot(PowerTransformer(method='yeo-johnson').fit_transform(df['SalePrice'].values.reshape(-1, 1)),label='Yeo-Johnson')
plt.xlim([-3,3])
plt.legend();

In [ ]:
# scale the inputs using the standard scaler
ss_inputs = StandardScaler()

X_train_scaled = ss_inputs.fit_transform(X_train)
X_test_scaled = ss_inputs.transform(X_test)
X_train_scaled

## Part 4 - Save the dataset

In [ ]:
# save the training and test parts of the dataset for later usage
X_train.to_csv("../data/housing_prices/X_train.csv"); y_train.to_csv("../data/housing_prices/y_train.csv");
X_test.to_csv("../data/housing_prices/X_test.csv"); y_test.to_csv("../data/housing_prices/y_test.csv");

## Part 5 - Feature selection methods
It is desirable to reduce the number of input variables to both reduce the computational cost of modeling and, in some cases, to improve the performance of the model.

We will go through examples of three different types of feature selection methods/approaches:
- filter methods: statistical-based feature selection methods that involve evaluating the relationship between each input variable and the target variable using statistics,
- wrapper methods: use different subsets of input features and select those features that result in the best performing model according to a performance metric,
- embedded/intrinsic methods: model-based built-in feature selection, meaning that the model will only include predictors that help maximize accuracy.

### Filter methods
Choose between the 10 most correlated features based on two different statistical test: Pearson correlation coefficient and mutual information measure.

In [ ]:
# import scikit-learn feature selection functions
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
# filter based feature selection function
def feature_selection(function, name, X, y):
    selection = SelectKBest(score_func=function, k='all').fit(X, y)
    feature_scores = pd.DataFrame({'features': X.columns.values, 'scores': selection.scores_})
    print(feature_scores.sort_values(by=['scores'], ascending=False))
    feature_scores.plot(x='features', kind='bar', title=name, figsize=(8, 6))
    return feature_scores

In [ ]:
f_regression_scores = feature_selection(f_regression, 'F regression - Pearson', X_train, y_train)

In [ ]:
# apply feature selection based on Mutual information measure
mutual_info_scores = feature_selection(mutual_info_regression, 'Mutual info', X_train, y_train)

In [ ]:
# compare obtained results
feature_scores = pd.DataFrame(index=relevant_inp[1:],columns=[['Pearson correlation', 'Mutual information']])
feature_scores['Pearson correlation'] = f_regression_scores.sort_values(by=['scores'], ascending=False).index
feature_scores['Mutual information'] = mutual_info_scores.sort_values(by=['scores'], ascending=False).index
feature_scores

### Wrapper methods
Find the most relevant (highest performing) features with 'forward-selection' approach for several models:
- linear regression,
- support vector machine/regressor,
- decision tree,
- random forest.

In [ ]:
# import wrapper algorithms
from mlxtend.feature_selection import SequentialFeatureSelector
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs

In [ ]:
from sklearn.linear_model import LinearRegression
# find best features for linear regression model
lr = LinearRegression()
sfs_lr = SequentialFeatureSelector(lr, k_features='best', forward=True, floating=False, scoring='neg_mean_absolute_error').fit(X_train, y_train)
fig = plot_sfs(sfs_lr.get_metric_dict(), kind='std_err', figsize=(20,6))
selected_features = X.columns[list(sfs_lr.k_feature_idx_)]
print(selected_features)

In [ ]:
# find best features for support vector regression model
from sklearn.svm import SVR
svr = SVR(kernel='linear')
sfs_svr = SequentialFeatureSelector(svr, k_features='best', forward=True, floating=False, scoring='neg_mean_absolute_error').fit(X_train, y_train)
plot_sfs(sfs_svr.get_metric_dict(), kind='std_err', figsize=(20,6))
selected_features = X.columns[list(sfs_svr.k_feature_idx_)]
print(selected_features)

In [ ]:
# find best features for decision tree regression model
from sklearn import tree
dtr = tree.DecisionTreeRegressor(criterion='absolute_error')
sfs_dtr = SequentialFeatureSelector(dtr, k_features='best', forward=True, floating=False, scoring='neg_mean_absolute_error', n_jobs=-1).fit(X_train, y_train)
fig = plot_sfs(sfs_dtr.get_metric_dict(), kind='std_err', figsize=(20,6))
selected_features = X.columns[list(sfs_dtr.k_feature_idx_)]
print(selected_features)

In [ ]:
# find best features for random forest regression model
from sklearn.ensemble import RandomForestRegressor
rfr = RandomForestRegressor(criterion='absolute_error', max_depth=10, random_state=0)
sfs_rfr = SequentialFeatureSelector(rfr, k_features='best', forward=True, floating=False, scoring='neg_mean_absolute_error', n_jobs=-1).fit(X_train, y_train)
fig = plot_sfs(sfs_rfr.get_metric_dict(), kind='std_err', figsize=(20,6))
selected_features = X.columns[list(sfs_rfr.k_feature_idx_)]
print(selected_features)

In [ ]:
feature_scores = pd.DataFrame(index=relevant_inp[1:],columns=[['LR', 'SVR', 'DT', 'RF']], data=0)
feature_scores.loc[relevant_inp[1:][list(sfs_lr.k_feature_idx_)],'LR'] = 1
feature_scores.loc[relevant_inp[1:][list(sfs_svr.k_feature_idx_)],'SVR'] = 1
feature_scores.loc[relevant_inp[1:][list(sfs_dtr.k_feature_idx_)],'DT'] = 1
feature_scores.loc[relevant_inp[1:][list(sfs_rfr.k_feature_idx_)],'RF'] = 1
feature_scores

### Embedded methods
Embedded feature selection methods are already incorporated into the intrinsical model procedures (more about those in the next few days).
We will evaluate the feature selection of two models: Lasso regression and random forest algorithm.

In [ ]:
# find features chosen by Lasso model
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

selection = SelectFromModel(Lasso(alpha=1))
selection.fit(X_train, y_train)

selected_features = X.columns[(selection.get_support())]
print(selected_features.values)

In [ ]:
# find features chosen by random forest model
from sklearn.ensemble import RandomForestRegressor

selection = SelectFromModel(RandomForestRegressor(n_estimators=100))
selection.fit(X_train, y_train)

selected_features = X.columns[(selection.get_support())]

print(selected_features.values)

# Iris dataset

The [Iris flower data set](https://archive.ics.uci.edu/ml/datasets/iris) is a multivariate data set introduced by the British statistician and biologist Ronald Fisher in his 1936 paper. The use of multiple measurements in taxonomic problems as an example of linear discriminant analysis. Two of the three species were collected in the Gaspé Peninsula "all from the same pasture, and picked on the same day and measured at the same time by the same person with the same apparatus".

The data set consists of 50 samples from each of three species of Iris (Iris setosa, Iris virginica and Iris versicolor). Four features were measured from each sample: the length and the width of the sepals and petals, in centimeters. Based on the combination of these four features, the **goal is to develop a model that is able to distinguish the species from each other**.

![title](../data/img/iris.png)

### Load dataset

In [ ]:
# load the dataset and print the first couple of rows
iris = pd.read_csv("../data/iris/Iris.csv", index_col='Id')
iris.head()

In [ ]:
# basic dataset properties
iris.info()

## Part 1 - Check basic dataset properties

In [ ]:
# basic dataset statistics
iris.describe()

In [ ]:
# basic statistcs per iris species
iris.groupby('Species').describe().transpose()

In [ ]:
# print just the means per species for easier visualisation
iris.groupby('Species').aggregate('mean')

In [ ]:
# count species (yeah, we know its three - just showing off)
iris["Species"].unique()

In [ ]:
# count the available data per each species
iris['Species'].value_counts()

## Part 2 - Exploratory data analysis (EDA)

In [ ]:
# scatter plot for sepal width/length per species - see if we can easily separate them through sepal differences
g = sns.relplot(data=iris, x='SepalLengthCm', y='SepalWidthCm', hue="Species", kind='scatter', s=100)
g.fig.set_size_inches((12,8))
sns.move_legend(g,'lower center',ncol=1, bbox_to_anchor=(.6,0.8), frameon=True);

In [ ]:
# add distributions of sepal length and width to help with visualization
g = sns.jointplot(x="SepalLengthCm", y="SepalWidthCm", data=iris, s=100, hue='Species')
g.fig.set_size_inches((8,8));

In [ ]:
# kde for sepals per species
g = sns.displot(data=iris, x="SepalLengthCm", y="SepalWidthCm", kind='kde', hue='Species')
g.fig.set_size_inches((12,8))
sns.move_legend(g,'center',ncol=1, bbox_to_anchor=(0.45,0.9), frameon=True)

In [ ]:
# scatter with distributions but for petals data this time
g = sns.jointplot(x="PetalLengthCm", y="PetalWidthCm", data=iris, s=100, hue='Species')
g.fig.set_size_inches((8,8));

In [ ]:
# kde for petals per species
g = sns.displot(data=iris, x="PetalLengthCm", y="PetalWidthCm", kind='kde', hue='Species')
g.fig.set_size_inches((12,8))
sns.move_legend(g,'center',ncol=1, bbox_to_anchor=(0.30,0.9), frameon=True)

In [ ]:
# look at an individual feature (petal length) through a box plot
plt.figure(figsize=(8,8))
sns.boxplot(x="Species", y="PetalLengthCm", data=iris);

In [ ]:
# add individual points on top of the boxplot
plt.figure(figsize=(8,8))
ax = sns.stripplot(x="Species", y="PetalLengthCm", data=iris, jitter=True, edgecolor="gray")
ax = sns.boxplot(x="Species", y="PetalLengthCm", data=iris, hue="Species", showfliers=False)

In [ ]:
# violin plot combines benefits of the previous two plots
# "fatter" parts represent denser regions, and "thiner" sparser regions of the data
plt.figure(figsize=(8,8))
sns.violinplot(x="Species", y="PetalLengthCm", data=iris, hue="Species");

In [ ]:
# check bivariate relations between each pair of features
g = sns.pairplot(iris, hue="Species", height=2.5);

In [ ]:
# calculate corellations between the available features
iris.corr(numeric_only=True)

In [ ]:
# plot correlations between features
corr = iris.corr(numeric_only=True)

mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True

# Want diagonal elements as well
mask[np.diag_indices_from(mask)] = False

plt.figure(figsize=(10,8))
sns.heatmap(corr,
            xticklabels=corr.columns,
            yticklabels=corr.columns,
            mask=mask, square=True, annot=True, linewidths=.5, cbar_kws={"shrink": .8});

## Part 3 - Process the training dataset

In [ ]:
# change species names to numbers (labels) in a manual way
iris = iris.replace({"Iris-setosa": 0, "Iris-versicolor": 1, "Iris-virginica": 2})
iris

In [ ]:
# split into inputs and outpus
X = iris[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']]
y = iris[["Species"]]

In [ ]:
# split into train-test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state = 1)

In [ ]:
# check the distribution of species in the training dataset
np.unique(y_train, return_counts=True)

In [ ]:
# check the distribution of species in the test dataset
np.unique(y_test, return_counts=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state = 1, stratify=y)

In [ ]:
# check the distribution of species in the training dataset
np.unique(y_train, return_counts=True)

In [ ]:
# check the distribution of species in the test dataset
np.unique(y_test, return_counts=True)

## Part 4 - Save the dataset

In [ ]:
# save the training and test parts of the dataset for later usage
X_train.to_csv("../data/iris/X_train.csv"); y_train.to_csv("../data/iris/y_train.csv");
X_test.to_csv("../data/iris/X_test.csv"); y_test.to_csv("../data/iris/y_test.csv");

### HANDS-ON: scale using Robust Scaler

Your assignement is to scale the input (X) and output (y) datasets (both train and test parts) by using the [Robust Scaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html), implemented in Sklearn.
Your task can be divided into the following steps:
- import the Robust Scaler,
- scale (fit and transform) the input dataset
- scale (fit and transform) the output dataset

Note: be careful, you have to use separate scaler objects ('ss_inputs' from above) for the inputs and the outpus. Additonally, you only **fit**-transform on the training part of the dataset, while the test part is only trasformed (data-leakage).

In [ ]:
# import the robust scaler
from sklearn.preprocessing import RobustScaler

In [ ]:
# scale the inputs using the robust scaler
rs_inputs = 

X_train_scaled = 
X_test_scaled = 

In [ ]:
# scale the outputs using the robust scaler
rs_outputs = 

y_train_scaled = 
y_test_scaled = 

# The end :)

## Bonus: automated EDA

In [ ]:
# %pip install sweetviz

In [ ]:
import sweetviz as sv
df = pd.read_csv('../data/housing_prices/housing.csv', index_col='Id')
analyze_report = sv.analyze(df)
analyze_report.show_html('../data/housing_prices/sweetviz_report.html', open_browser=False)

In [ ]:
# %pip install ydata_profiling

In [ ]:
# from ydata_profiling import ProfileReport

In [ ]:
# df = pd.read_csv('../data/housing_prices/housing.csv',index_col='Id')
# profile = ProfileReport(df,title='AMES Housing dataset')
# profile.to_file('../data/housing_prices/automated_report.html')



_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  

_Course: AI bootcamp - basic_  
_Notebook: 1_EDA_data_processing_  
_Instructors: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_  
_Intended user: XY_  
_Date: 1st January 1991_  

_References:_
- _Kaggle House Prices competition: https://www.kaggle.com/c/house-prices-advanced-regression-techniques_
- _Ames Housing Dataset (hosted on Kaggle): https://www.kaggle.com/datasets/prevek18/ames-housing-dataset_
- _Iris dataset: https://archive.ics.uci.edu/dataset/53/iris_
- _FISHER, R.A. (1936), THE USE OF MULTIPLE MEASUREMENTS IN TAXONOMIC PROBLEMS. Annals of Eugenics, 7: 179-188. https://doi.org/10.1111/j.1469-1809.1936.tb02137.x_
- _Numerical Python - NumPy: https://numpy.org/_
- _Python Data Analysis - pandas: https://pandas.pydata.org/_
- _Scientific Python - SciPy: https://scipy.org/_
- _Matplotlib: https://matplotlib.org/_
- _seaborn: https://seaborn.pydata.org/_
- _Scikit-learn Feature selection: https://scikit-learn.org/1.5/modules/feature_selection.html_
- _Scikit-learn Preprocessing data: https://scikit-learn.org/1.5/modules/preprocessing.html_
- _Scikit-learn Model selection and evaluation: https://scikit-learn.org/1.5/model_selection.html_
- _mlxtend data science tools library: https://pypi.org/project/mlxtend/0.1.7/_
- _Sweetviz: an open-source Python library for automated EDA, https://pypi.org/project/sweetviz/_
- _YData Profiling: package for data profiling, https://docs.profiling.ydata.ai/latest/_

_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_

